# Операция Химера: первый проверенный результат
Откройте notebook из папки practice или корня проекта. Установите requirements.txt в окружение выбранного ядра. Данные искусственные: из них нельзя делать вывод об активности. В Colab сначала загрузите chemistry.py, stepik_tasks.py и оба CSV в рабочую папку; установки выполняются только для открытых учебных данных.

In [ ]:
import sys
from pathlib import Path
import csv
import math
import rdkit
print(sys.executable, sys.version, rdkit.__version__)
candidates = [Path.cwd(), Path.cwd() / 'Stepik_course' / 'practice']
practice = next((p for p in candidates if (p / 'chemistry.py').exists()), None)
if practice is None:
    raise FileNotFoundError('Откройте notebook из practice или корня проекта')
sys.path.insert(0, str(practice))
from chemistry import standardize, descriptors, screen, curate_activity, similarity, optimize_conformer
from stepik_tasks import tanimoto, enrichment, z_prime

## 1. Сначала спрогнозируйте
Какая часть натрий ацетата останется? Сохранится ли ковалентный Cl в хлорбензоле? Запишите прогноз до запуска. Нейтральный parent не означает, что измерения соли и кислоты взаимозаменяемы.

In [ ]:
salt = standardize('CC(=O)[O-].[Na+]')
print(salt)
print(standardize('Clc1ccccc1'))
print(descriptors(salt.parent_smiles))
assert salt.parent_smiles == standardize('CC(=O)O').parent_smiles
assert 'Cl' in standardize('Clc1ccccc1').parent_smiles
assert math.isclose(descriptors('CCO')['MolWt'], 46.069, abs_tol=0.01)

## 2. Предупреждение и доказательство
Предскажите статусы этанола, катехола и неверной записи. Затем объясните, почему eligible не означает безопасен или активен.

In [ ]:
with (practice / 'molecules.csv').open(encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
results = [{'id': row['id'], **screen(row['smiles'])} for row in rows]
for row in results:
    print(row['id'], row['status'], row['reason'])
assert len(results) == len(rows)
assert next(r for r in results if r['id'] == 'M06')['status'] == 'review'
assert next(r for r in results if r['id'] == 'M09')['status'] == 'invalid'

## 3. Данные активности
Объясните каждое исключение. Числа вымышлены. R01 и R02 не являются независимыми подтверждениями.

In [ ]:
with (practice / 'activities.csv').open(encoding='utf-8') as handle:
    activities = [curate_activity(row) for row in csv.DictReader(handle)]
for row in activities:
    print(row['record_id'], row['status'], row.get('pIC50'), row['reason'])
assert [r['record_id'] for r in activities if r['status'] == 'included'] == ['R01', 'R02']

## 4. Представления и геометрия
Почему сходство 1 не доказывает измеренную активность? Почему оптимизация одного конформера не доказывает глобальный минимум?

In [ ]:
print('Танимото:', tanimoto({1,2,3}, {2,3,4}))
print('Morgan:', similarity('CCO', 'OCC'))
print('Z prime:', z_prime(100, 10, 5, 5))
print('EF:', enrichment(1000, 20, 50, 5))
print('Конформер:', optimize_conformer('CCCC'))

## 5. Самостоятельная часть
Добавьте новый валидный SMILES, пустую строку и структуру с ковалентным Br. Проверьте журнал. Напишите три предложения: какой вопрос решён; какой вывод пока недопустим; какой следующий контроль нужен. Для настоящего QSPR используйте отдельный train_esol.py и требования requirements-ml.txt, а не искусственные CSV.